# Lab notebook — pre-launch item difficulty estimation

**Project:** Levante QA / VLM panel  
**Location:** `tools/vlm-panel/`  
**Purpose:** Living record of experiments to estimate difficulty of **new / unscored** items (especially CAT tasks that need bank-scale `d`) before children see them.

Treat this like a paper lab notebook: append dated entries; do not rewrite history. When metrics change, add a new section and point at the artifact files under `out/`.

**How to update:** after each experiment, append a dated markdown cell + (optional) a code cell that reloads JSON/CSV from `out/`. Keep claims tied to file paths.

---

## Table of contents

1. [Question & framing](#1-question--framing)
2. [Tooling inventory](#2-tooling-inventory)
3. [Experiment track: age gradients](#experiment-track-age-gradients)
4. [Prompt history (from git)](#prompt-history-from-git)
5. [Chronology of work](#3-chronology-of-work)
6. [Key quantitative results](#4-key-quantitative-results)
7. [Conceptual learnings](#5-conceptual-learnings)
8. [Open questions / next](#6-open-questions--next)
9. [Reload live artifacts](#7-reload-live-artifacts)

## 1. Question & framing

### Goal

Before release, estimate how hard a **proposed new item** (or translation) will be for children — especially for **CAT** tasks (TROG, matrix reasoning, mental rotation, same-different) where the runtime needs a bank-scale IRT difficulty `d`.

### Scope we settled on

- Primary eval tasks: **TROG** + **vocab** (evaluable human data).
- Not yet shipping bank patches; building measurement + prediction tools first.
- Always use locale **`en-US`** for EN panels (bare `en` → `audio/en/` 404s → black preload).

### Two different “difficulty” targets

| Target | Meaning | Use |
|--------|---------|-----|
| `p_pred_child` | Predicted child pass rate | Triage / “about what % of kids get this?” |
| Bank `d` (GCS item bank) | IRT difficulty, **higher = harder** | CAT item selection |
| Bench `item_params` (Redivis) | Research Rasch/2PL export | Report-only; often **easiness-coded** (opposite sign vs bank) |

**Normal bank `d`:** fitted from large child response matrices (Rasch/2PL, often multigroup), then shipped in GCS. New items have no row in that matrix until field calibration.

**Working stance (2026-08-07):** For **new CAT items**, set **initial** bank `d` from hybrid `d_est` (panel → `p_pred` → features → bank scale) as a prior; always refit after child data. See §5 “CAT initial-`d` prior.”

## 2. Tooling inventory

| Piece | Path | Role |
|-------|------|------|
| Panel runner | `run_panel.mjs` + `panel_grid*.json` | Ungated VLM respondents × ages × models |
| Analyze | `analyze.mjs` | `p_vlm`, human join, calibrator → `p_pred_child` |
| Bench calibrator | `fit_bench_calibrator.mjs` | Fit on levante-bench trials |
| Hybrid `d_est` | `estimate_difficulty.mjs` | Map `p_pred` (+ TROG tags / vocab Zipf) → bank `d` |
| ICC `d_icc` | `fit_icc_difficulty.mjs` | Fit Rasch-with-guessing from age→θ panel trials |
| Age gradient check | `eval_age_gradient.mjs` | `med_p` by age, item spreads |
| TROG prompts | `cypress/support/agents/prompts/trogPrompts.ts` | Age-conditional system + `trogUserText` (imported by agent) |
| TROG agent | `cypress/support/agents/trogVlmAgent.ts` | Cypress decide loop; re-exports prompts |
| Persona | `cypress/support/persona/childPersona.ts` | Age/θ preamble + TROG mastery cues |
| Learnings | `LEARNINGS.md`, `RESULTS.md` | Operator docs |

**Panel formula (ungated):** VLM answers real UI → empirical `p_vlm` → monotonic calibrator → `p_pred_child`.

**Do not use** `QA_PERSONA_GATE=irt` for new items (needs existing `d`; collapses unscored items).

## Experiment track: age gradients

**Role in this notebook:** supporting method for bank-scale / ICC difficulty — not a separate research program.

**Why it matters:** CAT overall accuracy can look flat across ages (older kids get harder items). Our panel is **fixed** (every age sees the same items), so per-item \(P(\mathrm{correct}\mid\theta)\) *should* rise with age if VLMs behave like IRT kids. Empirically they were nearly flat (median item age spread ~0.10 vs ~0.21 Rasch-expected), which is why linked `d_icc` failed (ρ ≈ 0.05).

**What we tried**

| Approach | Outcome |
|----------|---------|
| Soft “act like a 6-year-old” persona | Already failed for age curves (`LEARNINGS.md`) — do not revive |
| Same adult TROG grammar checklist at all ages | Flattens curves (age-6 parses like an adult) |
| Age-conditional checklist (≤8 light / ≥10 full) + TROG mastery cues | Mini-eval **GO**: Δ med_p(13−6) 0.030 → **0.071**; item mean spread 0.086 → **0.152**; full-panel MAE p_pred **0.059** |

**How we measure**

1. Primary: respondent `med_p` by age → Δ(a12 − a6) via `eval_age_gradient.mjs` (was a13)
2. Item `max(p)−min(p)` across ages (mean/median spread)
3. Guardrail: full-panel MAE `p_pred` ≤ ~0.09
4. Downstream: `fit_icc_difficulty.mjs` ρ vs −p_pred / multivar `d_est`

**What “good” accuracy looks like (VLM ≠ child)**

Replay prints running `acc=correct/scored` per cell. That is **panel accuracy**, not child pass rates.

| Signal | Expectation |
|--------|-------------|
| EN age-6 VLM cell (end of bank) | Typically **~0.75–0.86** (historical median ≈ **0.81**); stronger cells can finish **~0.94+** |
| Mid-run (e.g. 63/70 ≈ 0.90) | **OK** — early TROG items are easier; totals often drift down by the end |
| Real age-6 kids (θ ≈ −2.0, n≈99) | Much lower than VLM totals — soft “act like 6” never closed that gap |
| What we care about | Age-6 ends **below** older ages (Δ med_p), item age-spreads, and MAE `p_pred` ≤ ~0.09 — not child-like raw accuracy |

Grid ages are now **`[6, 8, 10, 12]`** (was 13; θ₁₂ ≈ −0.22, n=104). Prefer capture-once + offline replay (`run_panel.mjs --capture-assets` then replay).

**Artifacts:** `out/age_grad_baseline_pre.json`, `out/age_grad_after.json`, `out/age_eval_gonogo.md`, `panel_grid_trog_age_eval.json`

**Status (2026-08-07):** GO for fuller EN force recollect with age-conditional prompts; ICC still weak until ages 8/10/12 are refreshed under the new grid. Append dated results under Chronology (§3) as this track continues.


## Prompt history (from git)

Canonical TROG prompt text now lives in [`cypress/support/agents/prompts/trogPrompts.ts`](../../cypress/support/agents/prompts/trogPrompts.ts) (extracted from `trogVlmAgent.ts` so wording can be versioned and reviewed without agent plumbing). Child persona preamble remains [`cypress/support/persona/persona_template.txt`](../../cypress/support/persona/persona_template.txt); TROG age-band mastery cues are appended in `childPersona.ts`.

Reconstructed from `git log` / `git show` on `trogVlmAgent.ts` (and persona template):

| Date | Commit | Prompt change |
|------|--------|----------------|
| 2026-05-31 | `3a232e5` *add trog* | **v0 baseline:** hear sentence → pick picture; short grammar reminder (word order, who/whom, negation, prepositions, clauses). Digit-only reply. No checklist, no `trogUserText`. |
| 2026-05-31 | `5bdefa1` / `d945985` | Same TROG system prompt. Age persona + IRT θ preamble added (`persona_template.txt`, `QA_PERSONA_ABILITY=irt`) — prepended in `cypress.config`, not in the TROG agent file. |
| 2026-07-30 | `027133a` | Child Twins panel plumbing; TROG `SYSTEM_PROMPT` still v0. |
| 2026-08-03 | `757cc01` *update model matrix, add results* | **v1 checklist:** five silent checks (agent/patient, negation scope, spatial, comparative, relative clauses). First **`trogUserText`** structure hints (negation, despite, spatial, size, chase/push). |
| 2026-08-06 | `4bcf684` *trog & vocab difficulties* | **v2 checklist:** passives explicit; spatial list + under/beneath; comparatives “named pair only”; embeddings example; **item 6** contrast connectives (despite/although/however/instead). Richer `trogUserText` (passive `by`, embeddings regex). Used for full EN force recollect → `d_est` ρ 0.532 → 0.637. |
| 2026-08-06 | *(working tree / lab)* age-conditional | **v3:** `SYSTEM_PROMPT_CHECKLIST` (v2) for age ≥10; **`SYSTEM_PROMPT_YOUNG`** (no checklist) for age ≤8; young runs skip structure hints. TROG mastery cues in persona by age band. Mini-eval GO (Δ med_p 0.030 → 0.071). |
| 2026-08-07 | *(lab)* | Prompts moved to `prompts/trogPrompts.ts`; agent re-exports for compatibility. |
| 2026-08-07 | *(lab / working tree)* | **v4 checklist + hints (ages ≥10 only):** HEAD-NOUN + tighter contrast + no agent-reversal on modifier `-ing`. Smoke 16 cells: duck ~0.13→**0.63**; despite ~0.20→**0.31**; Δ(12−6)=**0.101**; car/truck **0.00** (regression). |
| 2026-08-07 | *(lab / working tree)* | **v4.1 head-noun split:** `that/who` → keep required participants; participial → reject other noun doing Z. Remeasure 16 cells: car/truck still **0.00**; duck 0.63→**0.44**; despite **0.38**. **NO-GO** on car/truck wording. |
| 2026-08-07 | *(lab / working tree)* | **v4.2 two-noun relative tip:** user hint only when `the X that/who the Y`; drop broad that/who hint. Participial unchanged. Smoke: car/truck **0.06** (still BROKEN); duck **0.56**; ρ **0.53**. **NO-GO**. |
| 2026-08-07 | *(lab / working tree)* | **Freeze = v4 wording** (best duck) + `trog_preploc_car_truck_follow_drive` added to `known_issues.json`. Stop relative rewrites; de/es refresh on frozen prompt. |

**Design tension (still active):** checklist improves absolute TROG calibration (models were too hard on structure) but, applied at every age, flattens age/θ curves needed for ICC `d_icc`. v3/v4 keep the young/light split; v4 only deepens structure rules for older personas.

**Related persona note:** Soft “act like a 6-year-old” alone did not produce age curves (`LEARNINGS.md`). Operational mastery + age-conditional *task* scaffolding is the path we kept.

When editing prompts: change `trogPrompts.ts`, force-recollect affected grid cells, append a dated row here + Chronology metrics.

## 3. Chronology of work

### 2026-08 — Early framing

- Confirmed levante-qa **consumes** GCS bank `d` / persona θ; it does not fit IRT for new items.
- Best existing tool for unscored items: **VLM panel** → calibrator.
- GCS bank `d` and Redivis `item_params` are **different scales** (TROG overlap Pearson ≈ −0.39 historically).

### Affine → hybrid `d_est`

1. **v1** `estimate_difficulty.mjs`: affine `d_est = α + β·z` from `p_pred_child`.
   - TROG Spearman vs bank `d` weak (~0.24).
   - Vocab ranking stronger (~0.62); affine cannot beat pass-rate ranking ceiling.
2. **v2 hybrid:** ridge + Huber IRLS; features = `z` + TROG construction tags (`tagResidual`) / vocab Zipf.
   - TROG held-out **ρ_multivar ≈ 0.532** vs p-only ceiling **≈ 0.284** (beats ceiling).
   - Vocab: ranking ~0.61; Zipf helps MAE more than ranking.
3. Strengthened TROG prompts (`trogVlmAgent.ts`: passive, comparative, despite/however, embeddings).

### Limited prompt-eval recollect (8 cells)

- Grid: `panel_grid_trog_prompt_eval.json` (ages 8/10).
- Bug: `QA_LANGUAGE=en` → audio 404; fixed to **`en-US`**.
- n=8 too small/noisy to claim prompt lift (ceiling worsened 0.284 → 0.211 in that snapshot).

### Full EN TROG force recollect (32 cells)

- `panel_grid.json`, `--lang en-US --force` (~6h; muted/paused mid-run for Zoom, then resumed).
- Post: `post_en_full_recollect.sh` → analyze → fit calibrator → `estimate_difficulty` vs `d_est_trog_en_baseline_full.json`.

**After full recollect (2026-08-06):**

| Metric | baseline | after | Δ |
|--------|----------|-------|---|
| Spearman multivar | 0.532 | **0.637** | +0.105 |
| −p_pred ceiling | 0.284 | **0.471** | +0.187 |
| MAE multivar | 0.886 | **0.822** | −0.063 |
| MAE p_pred vs human | 0.076 | **0.063** | −0.013 |

Artifacts: `out/d_est_trog_en_report.md`, `out/trog_en_pred_after.json`.

### Idea: match new-item `p_pred` to similar `p_child`

- Reasonable for **triage** (same as p-only ranking).
- Insufficient for bank-scale CAT `d` when bank `d` ≉ reorder of pass rates (TROG Spearman(d_bank, −p_human) ≈ 0.43).

### Idea: variety of θs → ICC `d_icc`

- Implemented `fit_icc_difficulty.mjs`:  
  `P(correct|θ) = c + (1−c)·sigmoid(θ − d_icc)`  
  with θ from `age_task_ability.json`, then CV affine link to bank `d`.
- **Result (pre age-conditional prompts):** linked ρ ≈ **0.054**; raw ρ ≈ 0.21; −p_pred on same anchors ≈ 0.25; multivar `d_est` ≈ **0.637**.
- Diagnosis: **flat VLM×age curves** (median item age spread ~0.10 vs ~0.21 Rasch-expected given bank `d`). Fixed panel (all ages see same items) *should* show larger per-item age gradients than CAT overall accuracy.

### Age-conditional TROG child-likeness (2026-08-06 → 08-07)

**Hypothesis:** same adult grammar checklist at every age flattens age curves. Soft “act like a 6-year-old” already failed (`LEARNINGS.md`) — instead make **task prompt** age-conditional.

**Changes:**

1. `trogVlmAgent.ts`: age ≤8 → light prompt, no checklist / no structure `trogUserText` hints; age ≥10 → keep checklist.
2. `childPersona.ts`: TROG mastery cues by age band (operational, not cute roleplay).
3. Eval grid: `panel_grid_trog_age_eval.json` (2 models × ages 6/13 × 2 reps = 8 cells).

**Age-eval results:**

| | before | after |
|--|--------|-------|
| med_p age 6 | 0.894 | 0.818 |
| med_p age 13 | 0.924 | 0.889 |
| **Δ med_p(13−6)** | 0.030 | **0.071** |
| item mean age spread | 0.086 | **0.152** |
| Full-panel MAE p_pred | — | **0.059** (≤0.09) |

- One cell failed: `35flashlite_a6_r1`.
- **Verdict: GO** toward fuller EN force recollect (`out/age_eval_gonogo.md`).
- ICC on mixed panel still weak (ρ_cv ≈ 0.06) — only a6/a13 refreshed.

### 2026-08-07 — Known-issue triage + prompt v4 (postmod / despite)

**Context:** After capture/replay + `analyze.mjs --human-source=bench`, EN review listed three BROKEN items. Grid ages are now `[6,8,10,12]`.

#### Corpus / triage change

| Item | Action | Why |
|------|--------|-----|
| `trog_conjcoord_say_sunny_however_rain` | Added to [`known_issues.json`](known_issues.json); **suppressed from `review_*.csv` / `review_xlang_*.csv`** | Known broken for everyone (human p≈0.04, panel p≈0.02). Still on `screen_*.csv` with `KNOWN:` reason — not an actionable panel finding. |
| `trog_postmod_duck_following_turtle_walking` | Kept on review | See below |
| `trog_disjunctive_despite_noise_she_focus` | Kept on review | See below |

Wiring: `analyze.mjs` loads `known_issues.json` per task; known UIDs stay on the full screen, drop out of review triage.

#### Why the other two were BROKEN (not mis-keys)

| Item | p_vlm | p_human | Dominant VLM error |
|------|-------|---------|-------------------|
| duck … following … walking | 0.13 | 0.49 | Picks both animals on bridge (`duck-turtle-on-bridge`) instead of head-only (`duck-on-bridge`) — **relative / participial postmodifier** failure |
| despite noise … reading | 0.20 | 0.42 | Picks `…-writes` instead of `…-reads` — **main-clause activity** under a concessive |

#### Prompt change (v4)

General construction rules in `trogPrompts.ts` (no item names): HEAD-NOUN rule + tighter contrast-connective wording + stop agent-reversal hint on modifier `-ing`. Ages ≤8 unchanged. See Prompt history table.

**Run:** not yet — waiting on force replay / limited smoke before full 32-cell.

**Verdict:** triage corpus cleaned; prompt hypothesis ready to test.

**Next:** force replay (or smoke on relative_clause + disjunctive items) → re-analyze → compare duck/despite flags and MAE p_pred / age Δ.

### 2026-08-07 — Prompt v4 smoke replay (16 cells)

**Hypothesis:** HEAD-NOUN + tighter contrast rules lift duck/despite without item-specific coaching; age Δ stays healthy.

**Change:** v4 prompts (already in `trogPrompts.ts`). Grid: [`panel_grid_trog_v4_smoke.json`](panel_grid_trog_v4_smoke.json) (2 models × ages 6/8/10/12 × **2** repeats = 16).

**Run:** `node tools/vlm-panel/run_panel.mjs --force --replay --lang en-US --grid tools/vlm-panel/panel_grid_trog_v4_smoke.json` (~63 min). Log: `out/logs/v4_smoke_replay.log`.

**Metrics** (smoke cells only; pooled item correctness):

| Metric | Pre-v4 (prior panel) | v4 smoke |
|--------|----------------------|----------|
| duck postmod p | ~0.13 | **0.625** (10/16) |
| despite/noise p | ~0.20 | **0.31** (5/16) |
| Δ med_p(12−6) | ~0.07–0.09 | **0.101** |

**Verdict:** **GO** on head-noun (duck). Despite only a small lift (still near chance) — contrast rule weak or construction still hard. Age gradient OK.

**Analyze** (`--run-id-re` smoke 16 cells only — unfiltered disk mix still shows duck BROKEN from old r3/r4):

| Item | flag | p_vlm | p_human |
|------|------|-------|---------|
| duck postmod | **OK** | 0.625 | 0.487 |
| despite/noise | **HARD** | 0.313 | 0.419 |
| sunny (known) | BROKEN (suppressed) | 0.000 | 0.040 |
| car truck follow (new concern) | **BROKEN** | 0.000 | 0.617 |

EN smoke screen: B2/H14/C60; ρ difficulty **0.57**; CV MAE p_pred **0.083**. Artifacts: `out/report.md`, `out/screen_en.csv`, `out/review_en.csv` (from filtered run).

**Next:** full repeats=4 force vs despite iteration; inspect `trog_preploc_car_truck_follow_drive` (relative + reverse agent — head-noun family).

### 2026-08-07 — Prompt v4.1 (head-noun: keep required participants)

**Hypothesis:** v4’s “reject other noun doing Z” was too broad for `that/who` relatives; models dropped required participants (car/truck → car-only).

**Change:** `trogPrompts.ts` — split head-noun hints:
- **`that/who`:** outer action on head; **still include** nouns the relative requires (no head-only that drops them).
- **Participial postmod:** unchanged intent — reject other noun also doing the same main action (duck case).

**Evidence (v4 smoke):** car/truck keyed `car-truck-into-tunnel`; 12/16 picked `car-into-tunnel`; p 0.31→**0.00**.

**Run:** `panel_grid_trog_v4_smoke.json` force replay (~1h). Log: `out/logs/v4_1_smoke_replay.log`.

**Metrics (16 cells):**

| Item | v4 | v4.1 |
|------|----|------|
| duck | 0.625 | **0.438** (partial regress) |
| despite | 0.31 | **0.375** |
| car/truck | 0.00 | **0.00** (still 14/16 `car-into-tunnel`) |
| Δ med_p(12−6) | 0.101 | **0.101** |

**Verdict:** **NO-GO** on car/truck wording — models still drop the required truck. Duck softer than v4. Need a different relative-clause strategy (or accept as HARD and don’t over-fit).

**Next:** rethink `that/who` hint (maybe: resolve who-did-what inside the relative *and* keep all clause participants); avoid trading duck for car.

### 2026-08-07 — Prompt v4.2 (two-noun relative tip)

**Hypothesis:** Broad that/who “keep participants” hints don’t fix car/truck and hurt duck; a tip that fires only on `the X that/who the Y` will lift car/truck without trading duck.

**Change:** `trogPrompts.ts` — user hint only when two overt nouns in that/who relative (“both must appear; reject head-only”); drop broad that/who user hint; participial path unchanged; checklist bullet tightened to the two-noun case.

**Run:** `panel_grid_trog_v4_smoke.json` force replay. Log: `out/logs/v4_2_smoke_replay.log`. Analyze: `--run-id-re 'panel_trog_en_(35flashlite|36flash)_a(6|8|10|12)_r[12]$'`.

**Metrics (16 cells):**

| Item / metric | v4 | v4.1 | v4.2 |
|---------------|----|------|------|
| duck | 0.625 | 0.438 | **0.563** HARD |
| despite | 0.31 | 0.375 | **0.375** HARD |
| car/truck | 0.00 | 0.00 | **0.063** BROKEN (still mostly `car-into-tunnel`) |
| ρ difficulty | 0.57 | 0.59 | **0.53** |
| MAE p_pred | 0.083 | 0.08 | **0.08** |
| Δ med_p(12−6) | 0.101 | 0.101 | **0.096** |

**Verdict:** **NO-GO**. Car/truck essentially still broken; duck partially recovered vs v4.1 but below v4; ρ slipped. Stop iterating general relative wording — accept car/truck as known/HARD or try a non-prompt lever.

**Next:** mark car/truck known-issue *or* freeze prompt near best duck (v4) and full-force; don’t chase another head-noun rewrite.

### 2026-08-07 — Freeze EN TROG + parallel de/es + vocab

**Decision:** Stop relative-clause prompt chasing. Freeze prompt at **v4** (best duck smoke). Add `trog_preploc_car_truck_follow_drive` to [`known_issues.json`](known_issues.json) (still on `screen_*.csv`).

**Parallel work started:**
1. **de/es TROG force** on frozen v4 via `run_langs_trog.mjs` + [`panel_grid_trog_xlang_limited.json`](panel_grid_trog_xlang_limited.json) (16 cells/lang). Log: `out/logs/xlang_de_es_force.log`. Analyze/`review_xlang_*.csv` after finish.
2. **Vocab refresh** (existing panels, no recollect): analyze + `estimate_difficulty` → ρ_multivar **0.613**, −p_pred ceiling **0.661**, MAE **1.399** (unchanged vs prior). Artifacts: `out/report_vocab.md`, `out/d_est_vocab_en_*`.

**Verdict:** freeze landed; xlang running; vocab metrics stable (prompting not the lever).

**Next:** when de/es done → analyze bench + triage `review_xlang_{de,es}.csv`; EN full-force only if we want bank `d_est` refresh on frozen v4.

### 2026-08-07 — de/es TROG force complete

**Run:** `run_langs_trog.mjs` + `panel_grid_trog_xlang_limited.json` force de-DE then es-CO (16 cells each, live). ~3.7h. Log: `out/logs/xlang_de_es_force.log`. `worst_cell_exit=0`.

**Analyze** (`--human-source=bench`, full disk mix + fresh 3.x cells):

| Lang | resp | flags B/H/C | ρ diff | MAE p_pred |
|------|------|-------------|--------|------------|
| en | 88 | 4/12/14 | 0.65 | 0.07 |
| de | 69 | 2/13/16 | 0.64 | 0.08 |
| es | 64 | 2/13/12 | 0.62 | 0.08 |

**Triage:** duck still BROKEN on de (p≈0.11) and es (p≈0.22). Xlang strong |Δ|≥0.25: **de 1** (`trog_abovebelow_square_below_star` Δ≈−0.27); **es 0**. Artifacts: `out/review_{de,es}.csv`, `out/review_xlang_{de,es}.csv`.

**Verdict:** de/es panels refreshed on frozen v4; few translation-delta candidates (es clean; one spatial DE).

**Next:** optional EN full-force for `d_est`; spot-check DE square/star item.

### 2026-08-07 — Decision: accept hybrid `d_est` (not model-θ / ICC) for new-item difficulty

- **What:** For new / unscored items, primary bank-scale estimate = hybrid `d_est` from `p_pred` + features. Do **not** block on estimating model θ or ICC `d_icc`. Temp/token tweaks won’t create useful ability disparity.
- **Why / evidence:** ICC linked ρ ~0.06 vs hybrid ~0.64 (full EN recollect); see Conceptual learnings Q&A same day. Smoke-fit metrics on disk (~0.50) are not the performance ceiling.
- **Follow-up:** EN full-force on frozen v4 when we want `out/d_est_trog_en_metrics.json` restored to full-panel quality.

### 2026-08-07 — CAT initial-`d` prior (product stance)

- **What:** Treat hybrid `d_est` as the recommended **initial bank `d` prior** for new CAT items / translations (panel + calibrator + features), then overwrite with field IRT.
- **Why / evidence:** Better than arbitrary/midpoint initials; EN TROG ranking ρ≈0.6+ on anchors; still a screen (MAE~0.8–0.9). Full write-up in §5 Q&A same day.
- **Follow-up:** Wire into CAT/item-bank authoring when EN full-panel `d_est` is restored; keep “not ground truth” in operator docs.

## 4. Key quantitative results

### 4.1 Child pass-rate prediction (TROG EN)

| Snapshot | MAE p_vlm | MAE p_pred |
|----------|-----------|------------|
| Pre full recollect (`trog_en_pred_baseline.json`) | 0.108 | 0.076 |
| Post full recollect (`trog_en_pred_after.json`) | 0.104 | **0.063** |
| After age-eval mix (`trog_en_pred_age_eval.json` full screen) | 0.106 | **0.059** |

### 4.2 Bank-scale `d_est` (TROG EN)

| Snapshot | ρ multivar | ρ −p_pred | MAE multivar |
|----------|------------|-----------|--------------|
| Baseline full (`d_est_trog_en_baseline_full.json`) | 0.532 | 0.284 | 0.886 |
| Post full recollect (lab 2026-08-06; was `d_est_trog_en_metrics.json`) | **0.637** | **0.471** | **0.822** |
| Smoke-only overwrite 2026-08-07 (v4.2 `r[12]` screen; current metrics file) | 0.499 | 0.234 | 0.915 |

**Note:** Prefer the post-full-recollect row for “how well hybrid works.” Re-fit after EN full-force on frozen v4 to refresh `d_est_trog_en_metrics.json`. New-item use: ranking/triage, not CAT ground truth (see §5 Q&A).

Vocab EN (`d_est_vocab_en_metrics.json`): ρ_multivar ≈ 0.613; −p_pred ceiling ≈ 0.661 (features don’t beat p-only ranking).

### 4.3 ICC from θ grid (TROG EN)

| Metric | Value |
|--------|-------|
| ρ linked `d_icc_cv` | ~0.05–0.06 |
| ρ raw `d_icc` | ~0.21 |
| θ grid | 6→−2.01, 8→−0.91, 10→−0.44, **12→−0.22** (was 13→−0.15) |

### 4.4 Age gradient (mini-grid a6 vs a13; grid now uses a12)

See age-gradients track for expected VLM `acc=` ranges and child vs panel caveat. Artifacts: `age_grad_baseline_pre.json`, `age_grad_after.json`.

## 5. Conceptual learnings

1. **Best child-behavior predictor:** ungated panel + calibrator → `p_pred_child`.
2. **IRT gate / sim_child:** useless for new items without `d`.
3. **TROG:** models too hard on structure → checklist helped absolute error; applied at all ages → flat age curves.
4. **Vocab:** models too easy on rare words → Zipf shrink in analysis; prompting barely moves lexical ceiling.
5. **Don’t ship TROG `d_est` as CAT ground truth** yet; use panel for triage / pass rates; hybrid `d_est` is promising for ranking vs bank `d`.
6. **Same pass rate ≠ same bank `d`.** Neighbor-matching on `p` is triage, not IRT calibration.
7. **θ-grid ICC needs real age sensitivity** in the VLM; soft personas failed; age-conditional *task* scaffolding is the current bet.
8. **Operator:** resume by default; `--force` only after prompt changes; always `en-US`.

### Model tier as “ability” — Q&A (2026-08-06)

**Q:** Is it a good idea to rely on differences in model ability when we don’t really know much about either model?

**A:** As a kid-ability continuum: **no**. As an engineering spread knob: **yes, with limits**.

We don’t know *why* flash-lite misses what flash gets. It isn’t θ — it’s a different training mix, size, and failure modes. Treating “lite → flash” like “age 6 → 12” overclaims the science.

What we *do* know empirically (EN TROG 3.5-flash-lite vs 3.6-flash):

- They **differ in score** a lot (~0.83 vs ~0.97) — enough to keep the spread gate alive; lite-only respondent SD ≈ 0.03 is too flat.
- That spread is mostly useful for **ranking / `p_vlm`**, which still tracks kid pass rates (ρ difficulty ~0.65, MAE `p_pred` ~0.08).
- It is **not** useful for kid-like discrimination (rpb ρ was negative). Different models don’t separate items the way different children do.
- Blind spots (negation, spatial, …) can be **shared** across tiers — then “ability” variance doesn’t rescue you; **cross-language shift** is the better check.

**Stance:** keep two models for spread; don’t interpret lite/flash gaps as developmental. Prefer claims validated against humans (`p_pred`, ρ difficulty, BROKEN catch) and xlang deltas over “the stronger model got it so the item is easy.” If cutting cost, drop repeats or an age before dropping a model.

### Accept hybrid `d_est` for new items — Q&A (2026-08-07)

**Q:** Can we estimate θ for panel models and use that to calculate bank-scale `d` for new items? Would sampling params (tokens, temperature) create useful ability disparity?

**A:** θ-for-models is doable (MLE/EAP on bank-`d` anchors from each cell’s response vector) but **won’t unlock new-item `d`** until VLMs show kid-like `P(correct|ability)` curves. Today we *assign* child mean θ by age (`age_task_ability.json`) as a persona cue; `fit_icc_difficulty.mjs` already uses that grid and linked `d_icc` ρ stays ~**0.05–0.06**. Re-estimating model θ mostly relabels the same flat curves. Temperature / max tokens / thinking budget mainly add noise or don’t apply (TROG answers are digits); **model tier + age scaffolding** remain the real spread knobs.

**Accept hybrid instead:** for unscored items, use panel → `p_pred_child` → **hybrid `d_est`** (`estimate_difficulty.mjs`: logit `z` + TROG construction tags / vocab Zipf → bank `d`), not ICC-from-θ.

**How well (held-out anchors, not field-proven on brand-new items):**

| Signal | What you get |
|--------|----------------|
| EN TROG hybrid ρ vs bank `d` | **~0.64** after full recollect (best); ~0.50 if fit on v4.2 smoke-only screen — don’t use smoke metrics as the ceiling |
| Pass-rate alone (`−p_pred`) | ~0.28–0.47 (hybrid beats when tags help) |
| Absolute MAE on bank `d` | ~0.8–0.9 — triage / draft CAT, not final bank rows |
| Child pass-rate MAE `p_pred` | ~**0.06** post full EN recollect |
| Vocab ranking | `−p_pred` ceiling ~**0.66**; hybrid ≈0.61 (features help MAE more than order) |

**Stance:** hybrid is good enough to **screen and rank** new TROG items. Don’t ship as CAT ground truth without field calibration. Restore full-panel `d_est` with EN force on frozen v4 when we need the ~0.64 number back on disk (`out/d_est_trog_en_metrics.json` was overwritten by a smoke fit on 2026-08-07). Sampling knobs are not the path to better `d`.

### Hybrid `d_est` as CAT initial-`d` prior — Q&A (2026-08-07)

**Q:** Is panel → hybrid `d_est` a useful innovation for how initial `d` is set when testing a new item on a CAT task?

**A:** **Yes — as a better prior, not a replacement for field calibration.**

Status quo initials are often a guess, a band midpoint, or “looks like item X,” weakly tied to child performance. This pipeline gives, *before any child data*, (1) calibrated `p_pred_child` and (2) bank-scale rank via hybrid `d_est` from a panel that took the **real item UI**. EN TROG held-out ρ ≈ **0.6+** vs bank `d` is clearly better than chance for ordering a draft bank.

**Innovative when:** stimulus-faithful (same task kids see); maps onto the **deployed bank scale** so CAT can start nearer the right difficulty region; surfaces BROKEN / xlang deltas pre-launch.

**Not innovative / limits:** still a **screen** (MAE on `d` ~0.8–0.9; VLM blind spots can mis-rank); must be overwritten once real trials exist; vocab gains less from hybrid tags than from panel `p` alone.

**Practical CAT use:** set initial `d` ← hybrid `d_est` (optionally shrink toward bank mean if uncertain) → field/CAT → refit IRT. Real ops win over arbitrary initials for new items and translations — only if panel `d` is never treated as ground truth.



## 6. Open questions / next

- [x] **v4 smoke replay** (`panel_grid_trog_v4_smoke.json`, 16 cells) — duck → OK; despite → HARD.
- [x] Re-analyze smoke-only (`--run-id-re '…_r[12]$'`) — ρ≈0.57, MAE p_pred≈0.083.
- [x] **v4.1 re-smoke** — car/truck still 0/16; duck softened; **NO-GO**.
- [x] **v4.2 re-smoke** — car/truck **0.06** still BROKEN; duck 0.56; ρ 0.53. **NO-GO**. Stop head-noun rewrites.
- [x] Freeze = **v4 prompt** + car/truck in `known_issues.json`.
- [x] Vocab analyze + `d_est` refresh — ρ_multivar 0.613 / −p_pred 0.661 (stable).
- [x] **de/es TROG force** (`panel_grid_trog_xlang_limited.json`) — done; ρ≈0.62–0.64; es strong_delta=0; de 1 spatial candidate.
- [x] Spot-check `trog_abovebelow_square_below_star` DE vs EN — above/below foil; not a translation break.
- [x] Decision: **accept hybrid `d_est`** for new-item bank-scale ranking (not model-θ / ICC). Documented in §5.
- [ ] EN full-force on frozen v4 → restore full-panel `d_est_trog_en_metrics.json` (~ρ 0.64).
- [ ] Decide whether age ≤8 light prompt is too aggressive for age 8.
- [ ] Keep this notebook updated after each experiment.

---

### Entry template (copy for new dated sections)

```markdown
### YYYY-MM-DD — <title>

**Hypothesis:** …
**Change:** … (files)
**Run:** … (grid / command)
**Metrics:** … (table + artifact paths)
**Verdict:** …
**Next:** …
```

## 7. Reload live artifacts

Run the cells below to refresh tables from `out/` without editing the narrative above.

In [ ]:
from pathlib import Path
import json

OUT = Path("out")
if not OUT.exists():
    OUT = Path("tools/vlm-panel/out")

def load(name):
    p = OUT / name
    if not p.exists():
        print(f"missing: {p}")
        return None
    return json.loads(p.read_text())

files = [
    "d_est_trog_en_baseline_full.json",
    "d_est_trog_en_metrics.json",
    "trog_en_pred_baseline.json",
    "trog_en_pred_after.json",
    "trog_en_pred_age_eval.json",
    "d_icc_trog_en_metrics.json",
    "age_grad_baseline_pre.json",
    "age_grad_after.json",
    "d_est_vocab_en_metrics.json",
]
data = {f: load(f) for f in files}
print("Loaded", sum(v is not None for v in data.values()), "/", len(files), "from", OUT.resolve())

In [ ]:
def fmt(x, d=3):
    return "—" if x is None else f"{x:.{d}f}"

base = data.get("d_est_trog_en_baseline_full.json") or {}
cur = data.get("d_est_trog_en_metrics.json") or {}
print("=== TROG d_est vs bank d ===")
print(f"{'metric':28} {'baseline':>10} {'current':>10} {'Δ':>10}")
for k in ("spearman_multivar", "spearman_neg_p_pred", "mae_multivar"):
    b, c = base.get(k), cur.get(k)
    delta = None if b is None or c is None else c - b
    print(f"{k:28} {fmt(b):>10} {fmt(c):>10} {fmt(delta):>10}")

pb = data.get("trog_en_pred_baseline.json") or {}
pa = data.get("trog_en_pred_after.json") or {}
print("\n=== TROG p_pred MAE ===")
print(f"baseline MAE p_pred: {fmt(pb.get('mae_p_pred_vs_human'))}")
print(f"after full recollect: {fmt(pa.get('mae_p_pred_vs_human'))}")

ag0 = data.get("age_grad_baseline_pre.json") or {}
ag1 = data.get("age_grad_after.json") or {}
print("\n=== Age gradient a6 vs a13 ===")
print(f"Δ med_p before: {fmt(ag0.get('delta_med_p'))}")
print(f"Δ med_p after:  {fmt(ag1.get('delta_med_p'))}")

icc = data.get("d_icc_trog_en_metrics.json") or {}
print("\n=== ICC ===")
print(f"ρ d_icc_cv: {fmt(icc.get('spearman_d_icc_cv'))}")
print(f"ρ −p_pred:  {fmt(icc.get('spearman_neg_p_pred'))}")

---

*End of initial lab dump (2026-08-07). Append below.*